# TP2: Streaming ETL: Kafka → Spark → Parquet

**Level:** Introductory

**Objective:** Read events from Kafka, enrich with a static lookup table and write processed events to Parquet files.

This student notebook provides a runnable skeleton and helpful hints. Kafka broker must be available.

# TP2 Prerequisite: Start Kafka (Linux host only)

> **Important:** The commands below must be executed in a **terminal on the Linux host** that will run Kafka/Zookeeper.  
> **Do NOT run these commands in the Jupyter notebook.**  
> If you use Podman, replace `docker` with `podman` in every command.
---

## 1. Create a private network
```bash
docker network create kafka-net
```
*Creates an isolated network so Zookeeper and Kafka can communicate by name.*
---

## 2. Start Zookeeper
```bash
docker run -d --name zookeeper --network kafka-net \
  -p 2181:2181 \
  -v zookeeper_data:/bitnami/zookeeper \
  -e ALLOW_ANONYMOUS_LOGIN=yes \
  bitnamilegacy/zookeeper:latest
```
*Runs Zookeeper in the background; port 2181 is exposed on the host.*
---

## 3. Start Kafka broker
**Replace `localhost` with the host IP reachable by your clients.**
```bash
docker run -d --name kafka --network kafka-net \
  -p 9092:9092 \
  -e KAFKA_BROKER_ID=1 \
  -e KAFKA_ZOOKEEPER_CONNECT=zookeeper:2181 \
  -e ALLOW_PLAINTEXT_LISTENER=yes \
  -e KAFKA_LISTENERS=PLAINTEXT://0.0.0.0:9092 \
  -e KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://localhost:9092 \
  bitnamilegacy/kafka:3.5.1
```
*Kafka runs in background; clients must use `KAFKA_ADVERTISED_LISTENERS` address.*
---

## 4. Quick tests inside the Kafka container
**Create a topic**
```bash
docker exec -it kafka kafka-topics.sh --create --topic test-topic --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
```
**List topics**
```bash
docker exec -it kafka kafka-topics.sh --list --bootstrap-server localhost:9092
```
**Produce messages (interactive)**
```bash
docker exec -it kafka kafka-console-producer.sh --topic test-topic --bootstrap-server localhost:9092
```
*Type messages and press Enter. End with Ctrl+D.*

**Consume messages from the beginning**
```bash
docker exec -it kafka kafka-console-consumer.sh --topic test-topic --from-beginning --bootstrap-server localhost:9092
```
*Stop consuming with Ctrl+C.*

> If running producer/consumer from the notebook or another machine, use the **advertised listener** host/port (e.g., `10.192.0.13:9092`) in your Python code.
---

## 5. Cleanup (when finished)
```bash
docker stop kafka zookeeper
docker rm kafka zookeeper
docker network rm kafka-net
```
---

## Notes / troubleshooting
- If a client cannot connect, check `KAFKA_ADVERTISED_LISTENERS` points to a reachable IP and host firewall allows port 9092.  
- Podman users: replace `docker` with `podman`.  
- For this TP, simple configuration is enough; advanced setups are out of scope.

## Prerequisites
- Kafka broker accessible at `localhost:9092` (or update BROKER variable)
- Spark configured with Kafka package: `org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0` (set in `PYSPARK_SUBMIT_ARGS` or spark configuration)
- Python packages: `kafka-python`, `pandas`

## Quick help
### Quick Spark / DataFrame reminder (cheat sheet)
- Create SparkSession: `from pyspark.sql import SparkSession; spark = SparkSession.builder.appName("app").getOrCreate()`
- Read CSV: `spark.read.option("header",True).csv("path")`
- Read Parquet: `spark.read.parquet("path")`
- Select columns: `df.select("col1","col2")`
- Filter rows: `df.filter(df.col > 10)` or `df.where("col > 10")`
- Add/modify column: `df.withColumn("new", expr(...))` or `df.withColumn("new", df.col * 2)`
- Cast column: `df.withColumn("ts", col("ts").cast("timestamp"))`
- Join: `df1.join(df2, on="key", how="left")`
- Aggregations: `df.groupBy("key").agg(count("*").alias("n"), sum("amount").alias("total"))`
- Cache/Persist: `df.cache()` then trigger with an action like `df.count()`
- Explain plan: `df.explain(True)` to see logical/physical plans
- Convert to pandas (small results): `df.toPandas()`
- Write Parquet: `df.write.mode("overwrite").parquet("out/")`
- For streaming: `spark.readStream.format("kafka")...` and `df.writeStream...start()`
- Use `checkpointLocation` for streaming durability


# TP in the Notebook

Once Kafka is running on the Linux host, you can start the TP2 exercises inside the Jupyter notebook.
---

## Important notes for students

- Use `bootstrap_servers='localhost:9092'` (the advertised listener) in all Python Kafka producers and consumers.
- Do **not** try to start Kafka/Zookeeper from the notebook, the notebook only connects to the running Kafka broker.
- All Python code should use the provided `pyspark`, `kafka-python`, `pandas`, etc. libraries installed in the environment.
- For small TP exercises, read or produce a few messages to avoid blocking the kernel.

In [ ]:
# Cell 1: Configuration variables
BROKER = 'localhost:9092'
TOPIC = 'events_with_product'
print('Broker set to', BROKER, 'Topic:', TOPIC)

In [ ]:
from kafka import KafkaProducer, KafkaConsumer

producer = KafkaProducer(bootstrap_servers=BROKER)
producer.send('test-topic', b'Hello ISAE!')
producer.flush()

consumer = KafkaConsumer('test-topic', bootstrap_servers=BROKER, auto_offset_reset='earliest',  consumer_timeout_ms=3000)
for msg in consumer:
    print(msg.value.decode('utf-8'))


In [ ]:
# Cell 2: Create a small static lookup CSV (products)
import pandas as pd
products = pd.DataFrame({'product_id':[100,101,102],'product_name':['A','B','C'],'category':['cat1','cat2','cat1']})
products.to_csv('data/products.csv', index=False)
print('Wrote data/products.csv')

In [ ]:
%%bash
set -euo pipefail

# download Spark streaming jars into ~/jars
JARS_DIR="${HOME}/kafka_jars"
mkdir -p "$JARS_DIR"

# URLs (Spark 4.0.1 / Scala 2.13 example)
URL_SPARK_SQL="https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.13/4.0.1/spark-sql-kafka-0-10_2.13-4.0.1.jar"
#URL_KAFKA_CLIENTS="https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.2.1/kafka-clients-3.2.1.jar"
URL_KAFKA_CLIENTS="https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.4.0/kafka-clients-3.4.0.jar"
URL_SPARK_TOKEN="https://repo1.maven.org/maven2/org/apache/spark/spark-token-provider-kafka-0-10_2.13/4.0.1/spark-token-provider-kafka-0-10_2.13-4.0.1.jar"
URL_CP="https://repo1.maven.org/maven2/org/apache/commons/commons-pool2/2.12.1/commons-pool2-2.12.1.jar"

echo "Downloading jars to $JARS_DIR (skip if already present)..."
wget -nc -P "$JARS_DIR" "$URL_SPARK_SQL" "$URL_KAFKA_CLIENTS" "$URL_SPARK_TOKEN" "$URL_CP" || {
  echo "wget failed — trying curl fallback..."
  which curl >/dev/null 2>&1 && curl -fsSL -o "$JARS_DIR/$(basename $URL_SPARK_SQL)" "$URL_SPARK_SQL" || true
  which curl >/dev/null 2>&1 && curl -fsSL -o "$JARS_DIR/$(basename $URL_KAFKA_CLIENTS)" "$URL_KAFKA_CLIENTS" || true
  which curl >/dev/null 2>&1 && curl -fsSL -o "$JARS_DIR/$(basename $URL_SPARK_TOKEN)" "$URL_SPARK_TOKEN" || true
  which curl >/dev/null 2>&1 && curl -fsSL -o "$JARS_DIR/$(basename $URL_CP)" "$URL_CP" || true
}

echo "Listing files in $JARS_DIR:"
ls -l "$JARS_DIR"

In [ ]:
# Cell 3: Spark streaming skeleton (run in Spark-enabled kernel)
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
import os

USER_HOME = os.environ["HOME"]

jars = ",".join([
    f"{USER_HOME}/kafka_jars/spark-sql-kafka-0-10_2.13-4.0.1.jar",
    f"{USER_HOME}/kafka_jars/kafka-clients-3.4.0.jar",
    f"{USER_HOME}/kafka_jars/spark-token-provider-kafka-0-10_2.13-4.0.1.jar",
    f"{USER_HOME}/kafka_jars/commons-pool2-2.12.1.jar"
])

spark = (SparkSession.builder
    .appName("TP2_ETL")
    .config("spark.jars", jars)
    .getOrCreate())

schema = 'id INT, ts LONG, product_id INT, value DOUBLE'
raw = (spark.readStream.format('kafka')
       .option('kafka.bootstrap.servers', BROKER)
       .option('subscribe', TOPIC)
       .option('startingOffsets', 'earliest')
       .load())

json_df = raw.selectExpr('CAST(value AS STRING) as json_str')
# parse JSON, join with products and write to parquet (see exercise below)
print('Streaming skeleton ready')

## Exercises (to implement)
1. Implement a small Python producer (outside Spark) that sends JSON events with fields: `id`, `ts`, `product_id`, `value` to topic `events_with_product`.
2. Parse the JSON in Spark streaming (`from_json`) and join with `data/products.csv` to add `product_name` and `category`.
3. Write the processed stream to Parquet files partitioned by `category` with a `checkpointLocation`.
4. After enrichment, send processed JSON to another Kafka topic `processed_events`. If spark connector is not available, show how to do it with `kafka-python` in `foreachBatch`.

**Submission:** completed notebook and `output/parquet/` folder containing processed Parquet files.

In [ ]:
# Helpful hints
# - Use spark.read.csv('data/products.csv', header=True, inferSchema=True) to load the lookup table.
# - Use .withColumn((col('ts')/1000).cast('timestamp')) to convert epoch ms to timestamp if needed.
# - foreachBatch is a simple way to process each micro-batch like a normal DataFrame
# - Inside foreachBatch(fn) your function fn(batch_df, batch_id) receives a normal Spark DataFrame for that micro-batch.
# - Note: foreachBatch requires the function to accept two arguments: batch_df and batch_id
# - batch_df: the Spark DataFrame for the current micro-batch
# - batch_id: unique ID of the micro-batch (must be present)
# - You can convert rows to JSON and send them to Kafka with a KafkaProducer.
# - Pick a small checkpoint directory for the exercise: 'checkpoints/tp2'

In [ ]:
spark.stop()